# Type2 HFSS Import

이 노트북은 `examples/type2.toml`에서 STEP ledger를 만든 뒤 HFSS import를 실행하고, Ansys에서 열 수 있는 `.aedt` 경로와 imported ledger를 보여준다.

- STEP export: `entry/generate_type2_step.py`
- HFSS import: existing AEDT desktop에 붙어서 실행
- 결과물: `run/aedt/type2_step_import/type2_import.aedt`
- 기본값: GUI 켜짐, 기존 desktop attach, import 종료 후 자동 release 없음


In [1]:
from __future__ import annotations

import json
from pathlib import Path
from pprint import pprint
import subprocess
import sys


def require_repo_root() -> Path:
    result = subprocess.run(
        ["git", "rev-parse", "--show-toplevel"],
        check=True,
        capture_output=True,
        text=True,
    )
    root_text = result.stdout.strip()
    if not root_text:
        raise RuntimeError("git rev-parse --show-toplevel returned an empty path")
    repo_root = Path(root_text)
    if not repo_root.is_dir():
        raise RuntimeError(f"repo root does not exist: {repo_root}")
    return repo_root


REPO_ROOT = require_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from entry.generate_type2_step import export_type2_step_artifacts
from peetsfea.aedt import Hfss
from peetsfea.backend.pyaedt.type2_step_import_pipeline import import_type2_step_ledger_into_hfss

TYPE2_TOML_PATH = REPO_ROOT / "examples" / "type2.toml"
STEP_OUTPUT_DIR = REPO_ROOT / "run" / "step" / "type2"
STEP_LEDGER_PATH = STEP_OUTPUT_DIR / "type2_step_ledger.json"
OUTPUT_AEDT_PATH = REPO_ROOT / "run" / "aedt" / "type2_step_import" / "type2_import.aedt"
IMPORTED_LEDGER_PATH = REPO_ROOT / "run" / "aedt" / "type2_step_import" / "type2_imported_ledger.json"
DESIGN_NAME = "type2_step_import"
GUI_ENABLED = True
ATTACH_TO_EXISTING_DESKTOP = True
HFSS_SESSION = None


def open_or_reuse_hfss_session():
    global HFSS_SESSION
    if HFSS_SESSION is None:
        HFSS_SESSION = Hfss(
            project=None,
            design=DESIGN_NAME,
            non_graphical=not GUI_ENABLED,
            new_desktop=not ATTACH_TO_EXISTING_DESKTOP,
        )
    return HFSS_SESSION


print(f"repo root: {REPO_ROOT}")
print(f"type2 TOML: {TYPE2_TOML_PATH}")
print(f"STEP ledger: {STEP_LEDGER_PATH}")
print(f"AEDT output: {OUTPUT_AEDT_PATH}")
print(f"imported ledger: {IMPORTED_LEDGER_PATH}")
print(f"GUI enabled: {GUI_ENABLED}")
print(f"attach to existing desktop: {ATTACH_TO_EXISTING_DESKTOP}")


repo root: /home/harry/Projects/PythonProjects/peetsfea-main
type2 TOML: /home/harry/Projects/PythonProjects/peetsfea-main/examples/type2.toml
STEP ledger: /home/harry/Projects/PythonProjects/peetsfea-main/run/step/type2/type2_step_ledger.json
AEDT output: /home/harry/Projects/PythonProjects/peetsfea-main/run/aedt/type2_step_import/type2_import.aedt
imported ledger: /home/harry/Projects/PythonProjects/peetsfea-main/run/aedt/type2_step_import/type2_imported_ledger.json
GUI enabled: True
attach to existing desktop: True


## 1. Export Type2 STEP Ledger

이 셀은 `examples/type2.toml`에서 object-level STEP artifact와 `type2_step_ledger.json`을 만든다.


In [2]:
step_ledger = export_type2_step_artifacts(
    toml_path=TYPE2_TOML_PATH,
    output_dir=STEP_OUTPUT_DIR,
    ledger_path=STEP_LEDGER_PATH,
    seed=0,
)

print(f"source TOML: {step_ledger['source_toml_path']}")
print(f"output dir: {step_ledger['output_dir']}")
print(f"seed: {step_ledger['seed']}")
print(f"non-model object count: {len(step_ledger['non_model_objects'])}")
print(f"modeled object count: {len(step_ledger['modeled_objects'])}")


source TOML: /home/harry/Projects/PythonProjects/peetsfea-main/examples/type2.toml
output dir: /home/harry/Projects/PythonProjects/peetsfea-main/run/step/type2
seed: 0
non-model object count: 1
modeled object count: 1


## 2. Import Into Existing HFSS Desktop

이 셀은 기존 AEDT desktop에 붙어서 import/save만 수행한다. 셀 끝에서 `release_desktop()`을 호출하지 않으므로 desktop이 꺼지지 않아야 한다.


In [3]:
hfss_session = open_or_reuse_hfss_session()
imported_ledger = import_type2_step_ledger_into_hfss(
    hfss=hfss_session,
    step_ledger_path=STEP_LEDGER_PATH,
    output_aedt_path=OUTPUT_AEDT_PATH,
    imported_ledger_path=IMPORTED_LEDGER_PATH,
)

print(f"AEDT path: {imported_ledger['aedt_path']}")
print(f"imported ledger: {imported_ledger['imported_ledger_path']}")
print(f"non-model imported objects: {len(imported_ledger['non_model_objects'])}")
print(f"modeled imported objects: {len(imported_ledger['modeled_objects'])}")
print("HFSS desktop was not released by this cell.")


PyAEDT INFO: Python version 3.12.0 (main, Oct  3 2023, 01:27:23) [Clang 17.0.1 ].
PyAEDT INFO: PyAEDT version 0.25.1.
PyAEDT INFO: Initializing Desktop session.
PyAEDT INFO: AEDT version 2025.2.
PyAEDT INFO: New AEDT session is starting on gRPC port 41863.
PyAEDT INFO: Starting new AEDT gRPC session on port 41863.
PyAEDT INFO: Launching AEDT server with gRPC transport mode: TransportMode.UDS
PyAEDT INFO: Electronics Desktop started on gRPC port 41863 after 18.2 seconds.
PyAEDT INFO: AEDT installation Path /opt/ansys_inc/v252/AnsysEM
PyAEDT INFO: Connected to AEDT gRPC session on port 41863.
PyAEDT WARNING: Service Pack is not detected. PyAEDT is currently connecting in Insecure Mode.
PyAEDT WARNING: Please download and install latest Service Pack to use connect to AEDT in Secure Mode.
PyAEDT INFO: Project Project62 has been created.
PyAEDT INFO: Added design 'type2_step_import' of type HFSS.
PyAEDT INFO: AEDT objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elaps

## 3. Inspect What HFSS Imported

이 셀은 imported ledger를 읽어서 HFSS에서 생성된 object name들을 요약한다.


In [4]:
payload = json.loads(IMPORTED_LEDGER_PATH.read_text(encoding="utf-8"))

summary = {
    "aedt_path": payload["aedt_path"],
    "source_step_ledger_path": payload["source_step_ledger_path"],
    "non_model_count": len(payload["non_model_objects"]),
    "modeled_count": len(payload["modeled_objects"]),
}
pprint(summary)

for group_name in ("non_model_objects", "modeled_objects"):
    print()
    print(group_name)
    print("-" * len(group_name))
    for entry in payload[group_name]:
        print(f"object_id: {entry['object_id']}")
        print(f"  role: {entry['role']}")
        print(f"  model_state: {entry['model_state']}")
        print(f"  step_path: {entry['step_path']}")
        print(f"  imported_object_names: {entry['imported_object_names']}")


{'aedt_path': '/home/harry/Projects/PythonProjects/peetsfea-main/run/aedt/type2_step_import/type2_import.aedt',
 'modeled_count': 1,
 'non_model_count': 1,
 'source_step_ledger_path': '/home/harry/Projects/PythonProjects/peetsfea-main/run/step/type2/type2_step_ledger.json'}

non_model_objects
-----------------
object_id: type2_non_model_scene
  role: non_model_scene
  model_state: False
  step_path: /home/harry/Projects/PythonProjects/peetsfea-main/run/step/type2/type2_non_model_scene.step
  imported_object_names: ['SOLID', 'SOLID_1', 'SOLID_2', 'SOLID_3', 'SOLID_4', 'SOLID_5', 'SOLID_6']

modeled_objects
---------------
object_id: tx_rect_void_coil
  role: tx_single_coil
  model_state: True
  step_path: /home/harry/Projects/PythonProjects/peetsfea-main/run/step/type2/objects/tx_rect_void_coil.step
  imported_object_names: ['tx_copper_l0', 'SOLID_7']


## 4. Manual Release Without Closing Desktop

필요할 때만 아래 셀을 실행한다. `close_projects=False`, `close_on_exit=False`라서 project와 desktop을 닫지 않는 release만 요청한다.


In [5]:
# Run this only when you want to detach this notebook session.
if HFSS_SESSION is None:
    print("No HFSS session is currently stored in HFSS_SESSION.")
else:
    HFSS_SESSION.desktop_class.release_desktop(close_projects=False, close_on_exit=False)
    HFSS_SESSION = None
    print("Released notebook HFSS handle without closing projects or desktop.")


PyAEDT INFO: Desktop has been released.
Released notebook HFSS handle without closing projects or desktop.
